# Amazon ML Challenge 2026: Business Entity Resolution
## 01 - Exploratory Data Analysis (EDA) & Dataset Profiling

### Objective:
- Profile the reference dataset (`Source 1`) and target candidate datasets (`Source 2`, `Source 3`).
- Analyze dataset dimensions, schema, data types, missing values, and duplicate entity IDs.
- Inspect text lengths for `business_name` and `business_address`.
- Evaluate `country` distribution across Train and Test splits (handling open-set strings).
- Inspect `train_ground_truth.tsv` structure, singleton S1 entities, S1->S2 / S1->S3 match distributions, and multi-match percentages.

In [1]:
import os
import csv
import json
import sys
from collections import Counter, defaultdict
import statistics

# Add project root to sys.path
sys.path.append(os.path.abspath('..'))
from src.data_loader import DataLoader, Entity

## 1. Dataset Dimensions & Schema Overview

In [2]:
# Load pre-computed complete summary from EDA run
with open('../eda_summary.json', 'r', encoding='utf-8') as f:
    eda_stats = json.load(f)

print(f"{'Dataset':<15} | {'Rows':<12} | {'Cols':<6} | {'Missing Address':<16} | {'Unique Names':<14}")
print("-" * 75)
for key, data in eda_stats.items():
    if key == 'ground_truth':
        continue
    missing_addr = data['missing'].get('business_address', 0)
    missing_pct = f"{missing_addr:,} ({missing_addr/data['rows']*100:.2f}%)" if missing_addr else "0 (0.00%)"
    print(f"{key:<15} | {data['rows']:<12,} | {len(data['cols']):<6} | {missing_pct:<16} | {data['unique_names']:<14,}")

## 2. Text Characteristics (Business Name & Address Lengths)

In [3]:
print(f"{'Dataset':<15} | {'Name Mean/Median':<20} | {'Name Min/Max':<15} | {'Addr Mean/Median':<20} | {'Addr Min/Max':<15}")
print("-" * 90)
for key, data in eda_stats.items():
    if key == 'ground_truth':
        continue
    ns = data['name_stats']
    as_ = data['addr_stats']
    n_mm = f"{ns['mean']} / {ns['median']}"
    n_rng = f"{ns['min']} - {ns['max']}"
    a_mm = f"{as_['mean']} / {as_['median']}"
    a_rng = f"{as_['min']} - {as_['max']}"
    print(f"{key:<15} | {n_mm:<20} | {n_rng:<15} | {a_mm:<20} | {a_rng:<15}")

## 3. Country Distribution & Open-Set Analysis
Notice that the **training dataset** contains only `US` and `India`, whereas the **test dataset** contains `France` (~15%) in addition to `India` and `US`. 
Therefore, country must be treated as an open-set string field.

In [4]:
for key in ['train_s1', 'test_s1']:
    data = eda_stats[key]
    print(f"\nCountry distribution for {key} (Total: {data['rows']:,}):")
    for country, count in data['country_dist']:
        pct = (count / data['rows']) * 100
        print(f"  - {country:<10}: {count:>10,} ({pct:.2f}%)")

## 4. Ground Truth Analysis & S1 Matching Dynamics

In [5]:
gt = eda_stats['ground_truth']
print(f"Ground Truth Records (S1 entities): {gt['rows']:,}")
print(f"Total Ground Truth Matches:          {gt['total_matches']:,}")
print(f"  -> S1 -> S2 Matches:               {gt['s2_matches']:,} ({gt['s2_matches']/gt['total_matches']*100:.2f}%)")
print(f"  -> S1 -> S3 Matches:               {gt['s3_matches']:,} ({gt['s3_matches']/gt['total_matches']*100:.2f}%)")
print("-" * 50)
print(f"Zero Match (Singleton) S1 Entities:  {gt['zero_matches_count']:,} ({gt['pct_zero_matches']}%)")
print(f"Single Match S1 Entities:            {gt['single_match_count']:,} ({gt['pct_single_match']}%)")
print(f"Multiple Matches (>1) S1 Entities:   {gt['multi_matches_count']:,} ({gt['pct_multiple_matches']}%)")
print(f"Mean Matches per S1 Entity:          {gt['mean_matches']}")
print(f"Median Matches per S1 Entity:        {gt['median_matches']}")
print(f"Max Matches for a single S1 Entity:  {gt['max_matches']}")

print("\nMatch Count Distribution:")
for num_matches, count in gt['match_distribution']:
    pct = (count / gt['rows']) * 100
    bar = '#' * int(pct * 2)
    print(f"  {num_matches:>2} matches: {count:>8,} ({pct:>5.2f}%) {bar}")